In [1]:
import aisuite as ai
import json
import re

client = ai.Client(
    {
        "ollama":{
            "timeout": 100
        }   
    }
)

In [15]:

def generate_pp(client: ai.Client, question: str, file_path: str, schema: str, model: str) -> str:
    
    prompt = f"""
    You are a Python and Pandas assistant. Given the schema of a DataFrame and the user's question, write a Python code using pandas to process the DataFrame.

    Schema:
    {schema}

    User question:
    {question}

    file_path: 
    {file_path}

    Store the output in a result variable. Do not print the result. 
    
    Respond with the python only.
    Return your answer *strictly* in this format:

    <execute_python>
    # valid python code here
    </execute_python>

    Do not add explanations, only the tags and the code.
    """
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=1,
    )
    return response.choices[0].message.content.strip()

In [3]:
schema = """
id (INTEGER)
product_id (INTEGER)
product_name (STRING)
brand (STRING)
category (STRING)
color (STRING)
action (STRING)
qty_delta (INTEGER)
unit_price (FLOAT)
notes (TEXT)
ts (DATETIME)
"""

model = "ollama:llama3"

In [12]:
#question = "Write the code to order by unit_price"
#question = "Compute total value for red protucts for sales action"
question = "Compute total income (consider quantity) for Red color protucts for sale action"
file_path = "inventory_events_100.csv"

In [16]:
script = generate_pp(client, question, file_path, schema, model)
print(script)

<execute_python>
import pandas as pd

df = pd.read_csv('inventory_events_100.csv')

result = df[(df['color'] == 'Red') & (df['action'] == 'sale')]['qty_delta'].multiply(df['unit_price']).sum()
</execute_python>


In [17]:
match = re.search(r"<execute_python>([\s\S]*?)</execute_python>", script)
ns = {} 
if match:
    print("Found Python code:")
    script_filtered = match.group(1).strip()
    print(script_filtered)
    print("--"*10)
    exec(script_filtered, ns)  # Execute the generated script
    output = ns["result"]
    print(output)
else:
    print("No Python code found in the response.")

Found Python code:
import pandas as pd

df = pd.read_csv('inventory_events_100.csv')

result = df[(df['color'] == 'Red') & (df['action'] == 'sale')]['qty_delta'].multiply(df['unit_price']).sum()
--------------------
-6942.289999999999


## Improve

In [18]:
def refine_script(
    question: str,
    script: str,
    schema: str,
    model: str,
) -> tuple[str, str]:
    """
    Reflect on whether a script's *shown output* answers the question,
    and propose an improved SQL if needed.
    Returns (feedback, refined_sql).
    """
    prompt = f"""
You are a python (pandas) reviewer and refiner.

User asked:
{question}

Original python script:
{script}

Data Frame Columns:
{schema}

Step 1: Briefly evaluate if the python script OUTPUT fully answers the user's question.
Step 2: If improvement is needed, provide a refined python script.
If the original python script is already correct, return it unchanged.

Return STRICT JSON with two fields:
{{
  "feedback": "<1-3 sentences explaining the gap or confirming correctness>",
  "refined_sql": "<final python script to run>"
}}
"""
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=1,
    )

    content = response.choices[0].message.content

    try:
        obj = json.loads(content)
        feedback = str(obj.get("feedback", "")).strip()
        refined_script = str(obj.get("refined_sql", script)).strip()
        if not refined_script:
            refined_script = script
    except Exception:
        # Fallback if model doesn't return valid JSON
        feedback = content.strip()
        refined_script = script

    return feedback, refined_script

In [19]:
feedback, script_V2 = refine_script(
    question=question,
    script=script,   # <- comes from generate_sql() (V1)
    schema=schema, # <- we reuse the schema from section 3.1
    model=model
)

In [20]:
print(feedback)

Here's my feedback:

{
  "feedback": "The original Python script is close, but it doesn't explicitly compute total income for Red color products. The quantity change (qty_delta) might not be the actual quantity sold, which could lead to incorrect results.",
  "refined_sql": """
import pandas as pd

df = pd.read_csv('inventory_events_100.csv')

result = df[(df['color'] == 'Red') & (df['action'] == 'sale')].groupby('product_id')['qty_delta'].sum().multiply(df.loc[df.groupby('product_id')['action'].apply(lambda x: x.eq('sale').sum()), 'unit_price'].values).sum()
"""
}

In the refined script, I made a few changes to ensure that we're calculating the total income for Red color products correctly:

1. I added `groupby` and `.sum()` to aggregate the quantity change (qty_delta) by product_id. This way, we're considering the actual quantity sold.
2. I used `loc` to select the unit_price values for each product only when the action is 'sale'. This ensures that we're using the correct unit price 

In [23]:
print(script_V2)

<execute_python>
import pandas as pd

df = pd.read_csv('inventory_events_100.csv')

result = df[(df['color'] == 'Red') & (df['action'] == 'sale')]['qty_delta'].multiply(df['unit_price']).sum()
</execute_python>


## Improve with feedback

In [ ]:
def refine_script_external_feedback(
    question: str,
    script: str,
    output: str,
    schema: str,
    model: str,
) -> tuple[str, str]:
    """
    Reflect on whether a script's *shown output* answers the question,
    and propose an improved SQL if needed.
    Returns (feedback, refined_sql).
    """
    prompt = f"""
You are a python (pandas) reviewer and refiner.

User asked:
{question}

Original python script:
{script}

Data Frame Columns:
{schema}

Python script OUTPUT:
{output}

Step 1: Briefly evaluate if the python script OUTPUT fully answers the user's question.
Step 2: If improvement is needed, provide a refined python script.
If the original python script is already correct, return it unchanged.

Return STRICT JSON with two fields:
{{
  "feedback": "<1-3 sentences explaining the gap or confirming correctness. DO not include code here>",
  "refined_sql": "<final python script to run> Respond with the python only.
                   Return your answer *strictly* in this format:
                     <execute_python>
                     # valid python code here
                     </execute_python>"
}}
"""
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )

    content = response.choices[0].message.content

    try:
        obj = json.loads(content)
        feedback = str(obj.get("feedback", "")).strip()
        refined_script = str(obj.get("refined_sql", script)).strip()
        if not refined_script:
            refined_script = script
    except Exception:
        # Fallback if model doesn't return valid JSON
        feedback = content.strip()
        refined_script = script

    return feedback, refined_script

In [ ]:
feedback, refined_script = refine_script_external_feedback(
    question,
    script,
    output,
    schema,
    model,
)

In [ ]:
print(feedback)
print("")
print(refined_script)

In [ ]:
match = re.search(r"<execute_python>([\s\S]*?)</execute_python>", script)
ns = {} 
if match:
    print("Found Python code:")
    script_filtered = match.group(1).strip()
    print(script_filtered)
    print("--"*10)
    exec(script_filtered, ns)  # Execute the generated script
    output = ns["result"]
    print(output)
else:
    print("No Python code found in the response.")